# 3. Hybrid GraphRAG Querying & Portable Exports

Demonstrates `QueryBuilder`, NetworkX community detection & shortest paths, `HybridGraphRAGRetriever`, and lossless exports (`JSON`, sharded `JSONL`, sanitized `GraphML` & `GEXF`).

In [ ]:
import tempfile
from pathlib import Path

from knowledgereduce import (
    Claim,
    DuckDBParquetStore,
    Entity,
    Evidence,
    HybridGraphRAGRetriever,
    KnowledgeLayer,
    LayerData,
    QueryBuilder,
    Relation,
    ReliabilityRating,
    bounded_shortest_path,
    detect_communities,
)
from knowledgereduce.export import (
    export_gexf,
    export_graphml,
    export_layer_json,
    export_layer_sharded,
    import_layer_sharded,
)

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    store = DuckDBParquetStore(tmp_path / "store")
    layer_data = LayerData(
        layer=KnowledgeLayer(layer_id="demo", name="GraphRAG Demo"),
        entities=[
            Entity(entity_id="ent_nels", canonical_name="Nels Lindahl", entity_type="PERSON"),
            Entity(entity_id="ent_ch", canonical_name="Civic Honors", entity_type="ORG"),
        ],
        claims=[
            Claim(
                claim_id="clm_1",
                statement="Nels Lindahl authored Civic Honors in 2006.",
                tags=["civic", "book"],
                reliability=ReliabilityRating.VERIFIED,
                evidence=[Evidence(source_id="src_1", excerpt="authored Civic Honors")],
            )
        ],
        relations=[
            Relation(relation_id="r1", source_id="clm_1", target_id="ent_nels", relation_type="MENTIONS"),
            Relation(relation_id="r2", source_id="clm_1", target_id="ent_ch", relation_type="MENTIONS"),
        ],
    )
    store.save_layer(layer_data)

    hits = QueryBuilder(store, "demo").filter_tag("civic").execute_claims()
    graph = store.to_networkx("demo")
    print("Hits:", [c.claim_id for c in hits])
    print("Shortest path:", bounded_shortest_path(graph, "ent_nels", "ent_ch", max_hops=3))
    print("Communities:", detect_communities(graph))

    ctx = HybridGraphRAGRetriever(store).retrieve("demo", "Civic Honors author", top_k=2, hops=1)
    print("GraphRAG score:", ctx.results[0].score)

    export_layer_json(layer_data, tmp_path / "layer.json")
    shard_dir = export_layer_sharded(layer_data, tmp_path / "shards", shard_size=1)
    restored = import_layer_sharded(shard_dir)
    export_graphml(layer_data, tmp_path / "graph.graphml")
    export_gexf(layer_data, tmp_path / "graph.gexf")
    print("Sharded roundtrip relations:", len(restored.relations))